# B+ → π+ π+ π−: Cartesian QMI / isobar closure without CP

This notebook removes the CP fit completely and asks a simpler question:

> Can a one-dimensional Cartesian QMI S-wave recover an isobaric S-wave truth?

The toy is generated from the same isobaric model used in the CP closure, but
with **no charge dependence**. The scalar truth contains only the paper
$\sigma$ pole. In the fit the $\sigma$ is replaced by a QMI that interpolates
the real and imaginary parts directly and independently in $s=m^2$.

For this first diagnostic:

- only one $B^+$ sample is generated and fitted;
- the $\rho(770)$ coefficient is the global amplitude reference, fixed to $1+0i$;
- the global QMI coefficient is fixed to $1+0i$;
- **all QMI knots are free**, including the first knot;
- the QMI fit does **not** start from the isobaric truth: every knot starts at
  real part 1 and imaginary part 0;
- higher-wave coefficients start at their generated values but remain free
  except for the $\rho(770)$ reference;
- magnitude and phase are derived only after the fit, with uncertainties
  propagated from the Cartesian covariance matrix;
- no post-fit scale factor or phase shift is applied in the S-wave closure plots.

If this test fails, the problem is inside the QMI / dynamic-cache /
normalization / single-sample `FitSession` chain rather than the CP machinery.


In [ ]:
import numpy as np
import pandas as pd
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dataclasses import dataclass

from dalitzplotfitter import (
    DecayChannel,
    DecayModel,
    FitSession,
    GounarisSakurai,
    Parameter,
    QMI,
    RealImag,
    RelativisticBreitWigner,
    Resonance,
    enable_x64,
    generate_signal_toy,
)

enable_x64()

SEED = 20260905
N_EVENTS = 250_000
NORMALIZATION_RESOLUTION = 1500
QMI_INTERPOLATION = "linear"
TOY_METHOD = "accept-reject"

channel = DecayChannel("B+", ("pi+", "pi+", "pi-"))
M_PI = float(channel.daughter_masses[0])

print("Toy size:", N_EVENTS)
print("Normalization resolution:", NORMALIZATION_RESOLUTION)
print("QMI interpolation:", QMI_INTERPOLATION)
print("Toy method:", TOY_METHOD)


## Isobaric truth

We keep the CP-averaged Cartesian coefficients from the previous example and
drop all `dx/dy` terms. Thus each component has a single coefficient

\[
c_j = x_j + i y_j.
\]

The $\rho(770)$ is the amplitude reference.


In [ ]:
TRUTH_COEFFICIENTS = {
    "rho770":    dict(x= 1.000, y= 0.000),
    "omega782":  dict(x= 0.091, y=-0.007),
    "f2_1270":   dict(x= 0.291, y= 0.204),
    "rho1450":   dict(x=-0.223, y= 0.191),
    "rho3_1690": dict(x= 0.073, y=-0.045),
    "sigma":     dict(x=-0.485, y= 0.284),
}

def truth_coefficient(name):
    p = TRUTH_COEFFICIENTS[name]
    return RealImag(p["x"], p["y"])


In [ ]:
@dataclass(frozen=True)
class PaperSigmaPole:
    # LHCb Eq. (16): A_sigma(m) = 1 / (s_sigma - m^2)
    def __call__(self, mass, context):
        m = jnp.asarray(mass)
        pole = jnp.asarray(context.pole_mass) - 1j * jnp.asarray(context.pole_width)
        s_sigma = pole**2
        return 1.0 / (s_sigma - m**2)


def generator_components():
    return [
        Resonance(
            "rho770", (0, 2), truth_coefficient("rho770"),
            mass=0.7708, width=0.1534, spin=1,
            lineshape=GounarisSakurai(),
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "omega782", (0, 2), truth_coefficient("omega782"),
            mass=0.78265, width=0.00849, spin=1,
            lineshape=RelativisticBreitWigner(),
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "f2_1270", (0, 2), truth_coefficient("f2_1270"),
            mass=1.2755, width=0.1867, spin=2,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "rho1450", (0, 2), truth_coefficient("rho1450"),
            mass=1.465, width=0.400, spin=1,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "rho3_1690", (0, 2), truth_coefficient("rho3_1690"),
            mass=1.6888, width=0.161, spin=3,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "sigma", (0, 2), truth_coefficient("sigma"),
            mass=0.563, width=0.350, spin=0,
            lineshape=PaperSigmaPole(),
            resonance_radius=4.0, parent_radius=4.0,
        ),
    ]


generator = DecayModel(
    channel,
    generator_components(),
    normalize_components=True,
    normalization_method="gauss-legendre",
    #normalization_resolution=NORMALIZATION_RESOLUTION,
    #normalization_pair=(0, 2),
)

print("Generator normalization points:", generator.normalization_sample.size)


## Generate the isobaric toy

The toy is generated with the exact accept-reject reference sampler. The
candidate pools are evaluated directly in Dalitz invariants; four-momenta are
not retained because this closure uses only $s_{12}$, $s_{13}$ and $s_{23}$.


In [ ]:
toy = generate_signal_toy(
    generator,
    N_EVENTS,
    seed=SEED,
    method=TOY_METHOD,
    include_momenta=False,
)

print("Generated events:", toy.size)
print("Toy method:", TOY_METHOD)
print("weights:", np.unique(np.asarray(toy.weights)))


In [ ]:
# Toy Dalitz plot and invariant-mass projections before any fit.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)

axes[0].hist2d(
    np.asarray(toy.s13),
    np.asarray(toy.s23),
    bins=180,
)
axes[0].set_xlabel(r"$s_{13}$ [GeV$^2$]")
axes[0].set_ylabel(r"$s_{23}$ [GeV$^2$]")
axes[0].set_title("isobaric toy Dalitz plot")

for ax, values, label in [
    (axes[1], np.sqrt(np.asarray(toy.s13)), r"$m_{13}$"),
    (axes[2], np.sqrt(np.asarray(toy.s23)), r"$m_{23}$"),
]:
    ax.hist(values, bins=100, histtype="step")
    ax.set_xlabel(label + " [GeV]")
    ax.set_ylabel("events")

plt.show()


## Build truth-informed Cartesian QMI knots

The physical scalar amplitude that the QMI must reproduce is not the bare
$\sigma$ lineshape. It includes the generated complex coefficient and the
component normalization convention used by the `DecayModel`:

\[
A_S^{\rm truth}(m)
=
c_\sigma\,N_\sigma\,A_\sigma(m).
\]

We evaluate this physical amplitude at the QMI knots and retain its real and
imaginary parts for the representation-only comparison. These truth values are
not used as the fit starting point.

The first and last knots are placed at the exact kinematic boundaries of the
$\pi^+\pi^-$ pair so the QMI does not rely on endpoint clamping inside the
physical region.


In [ ]:
def component_by_name(model, name):
    for component in model.amplitude_model.components:
        if component.name == name:
            return component
    raise KeyError(name)


def scalar_truth_single_pair(masses):
    comp = component_by_name(generator, "sigma")
    coefficient = complex(np.asarray(comp.coefficient.value({})))
    scale = float(np.asarray(generator._component_scale(comp, {})))
    lineshape = comp.function.lineshape
    context = comp.function.context.resolve({})
    return coefficient * scale * np.asarray(
        lineshape(jnp.asarray(masses), context)
    )


PAIR_THRESHOLD = channel.daughter_masses[0] + channel.daughter_masses[2]
PAIR_MAXIMUM = channel.parent_mass - channel.daughter_masses[1]

QMI_KNOTS = (
    float(PAIR_THRESHOLD),
    0.400, 0.510, 0.630, 0.700, 0.770, 0.840, 0.900,
    0.990, 1.110, 1.210, 1.300, 1.400, 1.560, 1.740, 2.000,
    2.500, 3.000, 3.500,
    float(PAIR_MAXIMUM),
)

truth_knot_amplitudes = scalar_truth_single_pair(np.asarray(QMI_KNOTS))
truth_knot_real = np.real(truth_knot_amplitudes)
truth_knot_imaginary = np.imag(truth_knot_amplitudes)
truth_knot_magnitudes = np.abs(truth_knot_amplitudes)
truth_knot_phases = np.unwrap(np.angle(truth_knot_amplitudes))

print("QMI knots:", len(QMI_KNOTS))
print("physical range:", QMI_KNOTS[0], "to", QMI_KNOTS[-1], "GeV")
print()
print(
    f"{'m [GeV]':>9s} {'Re A_truth':>12s} {'Im A_truth':>12s} "
    f"{'|A_truth|':>12s} {'phase [rad]':>14s}"
)
for mass, real, imaginary, magnitude, phase in zip(
    QMI_KNOTS,
    truth_knot_real,
    truth_knot_imaginary,
    truth_knot_magnitudes,
    truth_knot_phases,
):
    print(
        f"{mass:9.4f} {real:12.6f} {imaginary:12.6f} "
        f"{magnitude:12.6f} {phase:14.6f}"
    )


### Representation-only check

Before fitting anything, construct a numerical Cartesian QMI directly from the
truth values at the knots. This isolates interpolation from the likelihood and
minimizer. The real and imaginary parts are interpolated independently in
$s=m^2$.

These truth knot values are **not** used as the fit starting point.


In [ ]:
truth_seed_qmi = QMI(
    knots=QMI_KNOTS,
    real_parts=tuple(float(value) for value in truth_knot_real),
    imaginary_parts=tuple(float(value) for value in truth_knot_imaginary),
    interpolation=QMI_INTERPOLATION,
)

qmi_context = component_by_name(generator, "sigma").function.context.resolve({})

mass_scan = np.linspace(QMI_KNOTS[0], QMI_KNOTS[-1], 3000)
truth_scan = scalar_truth_single_pair(mass_scan)
seed_scan = np.asarray(
    truth_seed_qmi(jnp.asarray(mass_scan), qmi_context)
)

relative_rms = np.sqrt(
    np.mean(np.abs(seed_scan - truth_scan)**2)
    / np.mean(np.abs(truth_scan)**2)
)

print(f"representation-only relative complex RMS = {relative_rms:.6e}")

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True, constrained_layout=True)

axes[0].plot(mass_scan, np.abs(truth_scan), label="isobar truth")
axes[0].plot(mass_scan, np.abs(seed_scan), "--", label="Cartesian QMI from truth knots")
axes[0].scatter(QMI_KNOTS, truth_knot_magnitudes, s=18, zorder=3)
axes[0].set_ylabel(r"$|A_S|$")
axes[0].legend()

axes[1].plot(mass_scan, np.unwrap(np.angle(truth_scan)), label="isobar truth")
axes[1].plot(
    mass_scan,
    np.unwrap(np.angle(seed_scan)),
    "--",
    label="Cartesian QMI from truth knots",
)
axes[1].scatter(QMI_KNOTS, truth_knot_phases, s=18, zorder=3)
axes[1].set_xlabel(r"$m(\pi^+\pi^-)$ [GeV]")
axes[1].set_ylabel("phase [rad]")
axes[1].legend()

plt.show()


## Fit model

All Cartesian QMI knot parameters are free. The global QMI coefficient is fixed
to $1+0i$, so the knot values themselves carry the physical S-wave amplitude.

The fit deliberately starts **away from the truth**:

- every QMI real part starts at `1.0`;
- every QMI imaginary part starts at `0.0`;
- higher-wave coefficients start at their generated values and remain free,
  except for the fixed $\rho(770)$ reference.

Magnitude and phase are not fit parameters. They are calculated after the fit
from the Cartesian result and its covariance matrix.


In [ ]:
def qmi_parameters():
    real_parts = []
    imaginary_parts = []
    for i in range(len(QMI_KNOTS)):
        real_parts.append(
            Parameter.dynamics(
                f"S_QMI.real[{i}]",
                1.0,
                owner="S_QMI",
                bounds=(-10.0, 10.0),
                step=0.01,
            )
        )
        imaginary_parts.append(
            Parameter.dynamics(
                f"S_QMI.imag[{i}]",
                0.0,
                owner="S_QMI",
                bounds=(-10.0, 10.0),
                step=0.01,
            )
        )
    return tuple(real_parts), tuple(imaginary_parts)


qmi_real_parts, qmi_imaginary_parts = qmi_parameters()
print("QMI fit start: all real parts = 1.0, all imaginary parts = 0.0")

qmi = QMI(
    knots=QMI_KNOTS,
    real_parts=qmi_real_parts,
    imaginary_parts=qmi_imaginary_parts,
    interpolation=QMI_INTERPOLATION,
)


def floating_coefficient(name):
    truth = TRUTH_COEFFICIENTS[name]
    reference = name == "rho770"
    x = Parameter.coefficient(
        f"{name}.x",
        truth["x"],
        owner=name,
        fixed=reference,
        step=0.01,
    )
    y = Parameter.coefficient(
        f"{name}.y",
        truth["y"],
        owner=name,
        fixed=reference,
        step=0.01,
    )
    return RealImag(x, y)


def fit_components():
    return [
        Resonance(
            "rho770", (0, 2), floating_coefficient("rho770"),
            mass=0.7708, width=0.1534, spin=1,
            lineshape=GounarisSakurai(),
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "omega782", (0, 2), floating_coefficient("omega782"),
            mass=0.78265, width=0.00849, spin=1,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "f2_1270", (0, 2), floating_coefficient("f2_1270"),
            mass=1.2755, width=0.1867, spin=2,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "rho1450", (0, 2), floating_coefficient("rho1450"),
            mass=1.465, width=0.400, spin=1,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "rho3_1690", (0, 2), floating_coefficient("rho3_1690"),
            mass=1.6888, width=0.161, spin=3,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "S_QMI", (0, 2), RealImag(1.0, 0.0),
            mass=1.0, width=0.1, spin=0,
            lineshape=qmi,
            normalize_component=False,
            resonance_radius=4.0, parent_radius=4.0,
        ),
    ]


fit_model = DecayModel(
    channel,
    fit_components(),
    normalize_components=True,
    normalization_method="gauss-legendre"
    # normalization_method="square-dalitz",
    # normalization_resolution=NORMALIZATION_RESOLUTION,
    # normalization_pair=(0, 2),
)

session = FitSession(fit_model, toy)

print("Number of fit parameters:", len(session.parameters))
print("Free parameters:", sum(not p.fixed for p in session.parameters))
print("rho770 fixed reference: 1+0i")
print("S_QMI global coefficient: fixed 1+0i")
print("all QMI knots: free")


## Component-normalization diagnostics

Before minimizing, inspect the normalization convention explicitly.

For each component we compute

[
I_i = \langle w,|F_i|^2\rangle,
qquad
N_i =
\begin{cases}
1/\sqrt{I_i}, & \text{if component normalization is enabled},\\
1, & \text{otherwise}.
\end{cases}
]

The conventional resonances should therefore have
(N_i^2 I_i=1), whereas the QMI deliberately has
`normalize_component=False` and must retain its raw physical scale.

We also compare the optimized normalization matrix used by the fit with a
direct recomputation on the same quadrature grid, and inspect the QMI diagonal
and interference terms at both the flat start and a truth-informed QMI point.
This isolates component normalization from the minimizer.


In [ ]:
from dalitzplotfitter.integration import normalization_matrix, matrix_normalization

normalization_sample = fit_model.normalization_sample
normalization_data = normalization_sample.as_dict()
normalization_weights = jnp.asarray(normalization_sample.weights)

normalization_probe_values = {
    parameter.name: float(parameter.value)
    for parameter in session.parameters
}

def component_norm_table(model, values):
    rows = []
    data = model.normalization_sample.as_dict()
    weights = jnp.asarray(model.normalization_sample.weights)
    for component in model.amplitude_model.components:
        raw = jnp.asarray(component.function(data, values))
        raw_integral = float(jnp.mean(weights * jnp.abs(raw) ** 2))
        scale = float(np.asarray(model._component_scale(component, values)))
        rows.append({
            "component": component.name,
            "normalize_component": (
                model.normalize_components
                if component.normalize_component is None
                else component.normalize_component
            ),
            "raw integral I": raw_integral,
            "scale N": scale,
            "scaled diagonal N^2 I": raw_integral * scale**2,
        })
    return pd.DataFrame(rows)

print("Generator component normalization")
display(component_norm_table(generator, {}))

print("Fit-model component normalization at the flat QMI start")
display(component_norm_table(fit_model, normalization_probe_values))

# The fixed higher-wave dynamics are identical in generator and fit model.
# Their normalization factors should therefore agree to numerical precision.
fixed_names = ("rho770", "omega782", "f2_1270", "rho1450", "rho3_1690")
scale_rows = []
for name in fixed_names:
    generator_component = component_by_name(generator, name)
    fit_component = component_by_name(fit_model, name)
    generator_scale = float(np.asarray(generator._component_scale(generator_component, {})))
    fit_scale = float(np.asarray(
        fit_model._component_scale(fit_component, normalization_probe_values)
    ))
    scale_rows.append({
        "component": name,
        "generator scale": generator_scale,
        "fit scale": fit_scale,
        "relative difference": (
            (fit_scale - generator_scale) / generator_scale
            if generator_scale != 0.0 else np.nan
        ),
    })

print("Generator vs fit normalization factors for the fixed waves")
display(pd.DataFrame(scale_rows))

# Compare the exact quadrature samples used by generator and fit model.
generator_norm = generator.normalization_sample
fit_norm = fit_model.normalization_sample
print("normalization sample sizes:", generator_norm.size, fit_norm.size)
if generator_norm.size == fit_norm.size:
    print(
        "max |Delta s13| =",
        float(np.max(np.abs(
            np.asarray(generator_norm.s13) - np.asarray(fit_norm.s13)
        ))),
    )
    print(
        "max |Delta s23| =",
        float(np.max(np.abs(
            np.asarray(generator_norm.s23) - np.asarray(fit_norm.s23)
        ))),
    )
    print(
        "max |Delta weight| =",
        float(np.max(np.abs(
            np.asarray(generator_norm.weights) - np.asarray(fit_norm.weights)
        ))),
    )

def direct_scaled_matrix(model, values):
    sample = model.normalization_sample
    data = sample.as_dict()
    weights = jnp.asarray(sample.weights)
    columns = []
    for component in model.amplitude_model.components:
        raw = jnp.asarray(component.function(data, values))
        scale = model._component_scale(component, values)
        columns.append(raw * scale)
    components = jnp.stack(columns, axis=1)
    return normalization_matrix(components, weights)

cache_matrix_start = np.asarray(
    session.signal_cache.normalization_matrix(normalization_probe_values)
)
direct_matrix_start = np.asarray(
    direct_scaled_matrix(fit_model, normalization_probe_values)
)

print(
    "max |M_cache - M_direct| at start =",
    float(np.max(np.abs(cache_matrix_start - direct_matrix_start))),
)

qmi_index = next(
    i
    for i, component in enumerate(fit_model.amplitude_model.components)
    if component.name == "S_QMI"
)

def print_qmi_matrix_row(matrix, label):
    rows = []
    for j, component in enumerate(fit_model.amplitude_model.components):
        value = matrix[qmi_index, j]
        rows.append({
            "with component": component.name,
            "Re M_Qj": float(np.real(value)),
            "Im M_Qj": float(np.imag(value)),
            "|M_Qj|": float(np.abs(value)),
        })
    print(label)
    display(pd.DataFrame(rows))

print_qmi_matrix_row(cache_matrix_start, "QMI normalization/interference row at start")
print("M_QQ(start) =", float(np.real(cache_matrix_start[qmi_index, qmi_index])))

# Replace only the QMI knots by the physical sigma truth sampled at the knots.
truth_like_values = dict(normalization_probe_values)
for parameter, value in zip(qmi_real_parts, truth_knot_real):
    truth_like_values[parameter.name] = float(value)
for parameter, value in zip(qmi_imaginary_parts, truth_knot_imaginary):
    truth_like_values[parameter.name] = float(value)

cache_matrix_truth = np.asarray(
    session.signal_cache.normalization_matrix(truth_like_values)
)
direct_matrix_truth = np.asarray(
    direct_scaled_matrix(fit_model, truth_like_values)
)

print(
    "max |M_cache - M_direct| at truth-like QMI point =",
    float(np.max(np.abs(cache_matrix_truth - direct_matrix_truth))),
)
print_qmi_matrix_row(
    cache_matrix_truth,
    "QMI normalization/interference row at truth-like knot values",
)

sigma_coefficient = complex(
    np.asarray(component_by_name(generator, "sigma").coefficient.value({}))
)
expected_sigma_contribution_norm = abs(sigma_coefficient) ** 2
qmi_truth_diagonal = float(np.real(cache_matrix_truth[qmi_index, qmi_index]))

print("M_QQ(truth-like) =", qmi_truth_diagonal)
print("|c_sigma|^2       =", expected_sigma_contribution_norm)
print(
    "relative QMI-vs-sigma diagonal difference =",
    (qmi_truth_diagonal - expected_sigma_contribution_norm)
    / expected_sigma_contribution_norm,
)

coefficients_start = jnp.asarray([
    component.coefficient.value(normalization_probe_values)
    for component in fit_model.amplitude_model.components
])
direct_total_start = float(
    matrix_normalization(coefficients_start, jnp.asarray(direct_matrix_start))
)
cache_total_start = float(
    session.signal_cache.normalization(normalization_probe_values)
)
print("total normalization at start (direct) =", direct_total_start)
print("total normalization at start (cache)  =", cache_total_start)

coefficients_truth = jnp.asarray([
    component.coefficient.value(truth_like_values)
    for component in fit_model.amplitude_model.components
])
direct_total_truth = float(
    matrix_normalization(coefficients_truth, jnp.asarray(direct_matrix_truth))
)
cache_total_truth = float(
    session.signal_cache.normalization(truth_like_values)
)
print("total normalization truth-like (direct) =", direct_total_truth)
print("total normalization truth-like (cache)  =", cache_total_truth)


## Baseline fit

The QMI starts from a flat Cartesian configuration, with every knot equal to
$1+0i$. This is a genuine closure test of the minimizer and QMI likelihood,
rather than a fit initialized on the truth.

Higher-wave coefficients still start at their generated values, with the
$\rho(770)$ fixed as the global amplitude reference.


In [ ]:
start_values = {
    parameter.name: float(parameter.value)
    for parameter in session.parameters
    if not parameter.fixed
}

result = session.fit(
    start_values=start_values,
    simplex=False,
    strategy=1,
    hesse=False,
    tolerance=1e-4,
    verbose=3,
)

fit_values = session.print_result(result)

print()
print("valid:", bool(result.valid))
print("EDM:", float(result.fmin.edm))
print("nfcn:", int(result.nfcn))


## Direct S-wave closure

No arbitrary scaling and no phase offset are applied below. Magnitude and phase
are derived from the fitted Cartesian QMI. Their uncertainties use the full
$2\times2$ covariance block of the real and imaginary parts at each knot.


In [ ]:
def resolved_qmi(qmi_model, values):
    return QMI(
        knots=qmi_model.knots,
        real_parts=tuple(
            float(parameter.resolve(values))
            if hasattr(parameter, "resolve") else float(parameter)
            for parameter in qmi_model.real_parts
        ),
        imaginary_parts=tuple(
            float(parameter.resolve(values))
            if hasattr(parameter, "resolve") else float(parameter)
            for parameter in qmi_model.imaginary_parts
        ),
        interpolation=qmi_model.interpolation,
    )


def fitted_qmi_single_pair(masses, values):
    comp = component_by_name(fit_model, "S_QMI")
    context = comp.function.context.resolve(values)
    dynamics = np.asarray(
        resolved_qmi(qmi, values)(jnp.asarray(masses), context)
    )
    coefficient = complex(np.asarray(comp.coefficient.value(values)))
    scale = float(np.asarray(fit_model._component_scale(comp, values)))
    return coefficient * scale * dynamics


def polar_uncertainties(real_name, imaginary_name, real, imaginary):
    radius = np.hypot(real, imaginary)
    if result.covariance is None or radius <= np.finfo(float).eps:
        return np.nan, np.nan

    var_real = float(result.covariance[real_name, real_name])
    var_imaginary = float(result.covariance[imaginary_name, imaginary_name])
    cov_real_imaginary = float(result.covariance[real_name, imaginary_name])

    var_magnitude = (
        real**2 * var_real
        + imaginary**2 * var_imaginary
        + 2.0 * real * imaginary * cov_real_imaginary
    ) / radius**2
    var_phase = (
        imaginary**2 * var_real
        + real**2 * var_imaginary
        - 2.0 * real * imaginary * cov_real_imaginary
    ) / radius**4
    return np.sqrt(max(var_magnitude, 0.0)), np.sqrt(max(var_phase, 0.0))


fit_scan = fitted_qmi_single_pair(mass_scan, fit_values)
fit_knot_real = np.asarray(
    [float(parameter.resolve(fit_values)) for parameter in qmi_real_parts]
)
fit_knot_imaginary = np.asarray(
    [float(parameter.resolve(fit_values)) for parameter in qmi_imaginary_parts]
)
fit_knot_amplitudes = fit_knot_real + 1j * fit_knot_imaginary
fit_knot_magnitudes = np.abs(fit_knot_amplitudes)

polar_errors = [
    polar_uncertainties(real_parameter.name, imaginary_parameter.name, real, imaginary)
    for real_parameter, imaginary_parameter, real, imaginary in zip(
        qmi_real_parts,
        qmi_imaginary_parts,
        fit_knot_real,
        fit_knot_imaginary,
    )
]
fit_knot_magnitude_errors = np.asarray([item[0] for item in polar_errors])
fit_knot_phase_errors = np.asarray([item[1] for item in polar_errors])

# Compare phases on the physical branch closest to the unwrapped truth.
truth_phase = np.unwrap(np.angle(truth_scan))
phase_difference = np.angle(fit_scan * np.conj(truth_scan))
fit_phase_aligned = truth_phase + phase_difference
aligned_knot_phases = truth_knot_phases + np.angle(
    fit_knot_amplitudes * np.conj(truth_knot_amplitudes)
)

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True, constrained_layout=True)

axes[0].plot(mass_scan, np.abs(truth_scan), label="isobar truth")
axes[0].plot(mass_scan, np.abs(fit_scan), "--", label="Cartesian QMI fit")
axes[0].errorbar(
    QMI_KNOTS,
    fit_knot_magnitudes,
    yerr=fit_knot_magnitude_errors,
    fmt="o",
    markersize=4,
    label="derived fitted knots",
)
axes[0].set_ylabel(r"$|A_S|$")
axes[0].legend()

axes[1].plot(mass_scan, truth_phase, label="isobar truth")
axes[1].plot(
    mass_scan,
    fit_phase_aligned,
    "--",
    label=r"Cartesian QMI fit (same $2\pi$ branch)",
)
axes[1].errorbar(
    QMI_KNOTS,
    aligned_knot_phases,
    yerr=fit_knot_phase_errors,
    fmt="o",
    markersize=4,
    label=r"derived fitted knots (same $2\pi$ branch)",
)
axes[1].set_xlabel(r"$m(\pi^+\pi^-)$ [GeV]")
axes[1].set_ylabel("phase [rad]")
axes[1].legend()

plt.show()

complex_rms = np.sqrt(
    np.mean(np.abs(fit_scan - truth_scan)**2)
    / np.mean(np.abs(truth_scan)**2)
)
phase_rms = np.sqrt(np.mean(phase_difference**2))
phase_max = np.max(np.abs(phase_difference))

print(f"post-fit relative complex RMS = {complex_rms:.6e}")
print(f"wrapped phase-difference RMS  = {phase_rms:.6e} rad")
print(f"max |wrapped phase difference| = {phase_max:.6e} rad")


## Knot-by-knot closure

The fitted parameters and pulls are reported first in Cartesian coordinates.
Magnitude and phase are derived quantities. Their errors are propagated with
the complete real/imaginary covariance block at each knot. If the fitted
magnitude is compatible with zero, the linearized phase uncertainty is not a
reliable description and should be replaced by a two-dimensional confidence
region in the complex plane.


In [ ]:
rows = []
for i, (
    mass,
    real_parameter,
    imaginary_parameter,
    real_truth,
    imaginary_truth,
    magnitude_truth,
    phase_truth,
) in enumerate(
    zip(
        QMI_KNOTS,
        qmi_real_parts,
        qmi_imaginary_parts,
        truth_knot_real,
        truth_knot_imaginary,
        truth_knot_magnitudes,
        truth_knot_phases,
    )
):
    real_fit = float(real_parameter.resolve(fit_values))
    imaginary_fit = float(imaginary_parameter.resolve(fit_values))
    real_error = float(result.errors[real_parameter.name])
    imaginary_error = float(result.errors[imaginary_parameter.name])
    magnitude_fit = np.hypot(real_fit, imaginary_fit)
    phase_fit = phase_truth + np.angle(
        (real_fit + 1j * imaginary_fit)
        * np.conj(real_truth + 1j * imaginary_truth)
    )
    magnitude_error, phase_error = polar_uncertainties(
        real_parameter.name,
        imaginary_parameter.name,
        real_fit,
        imaginary_fit,
    )
    rows.append({
        "knot": i,
        "m [GeV]": mass,
        "Re truth": real_truth,
        "Re fit": real_fit,
        "Re err": real_error,
        "Re pull": (real_fit - real_truth) / real_error,
        "Im truth": imaginary_truth,
        "Im fit": imaginary_fit,
        "Im err": imaginary_error,
        "Im pull": (imaginary_fit - imaginary_truth) / imaginary_error,
        "mag truth": magnitude_truth,
        "mag fit": magnitude_fit,
        "mag err": magnitude_error,
        "mag pull": (
            (magnitude_fit - magnitude_truth) / magnitude_error
            if magnitude_error > 0.0 else np.nan
        ),
        "phase truth": phase_truth,
        "phase fit": phase_fit,
        "phase err": phase_error,
        "phase pull": (
            (phase_fit - phase_truth) / phase_error
            if phase_error > 0.0 else np.nan
        ),
    })

qmi_table = pd.DataFrame(rows)
display(qmi_table)


## Higher-wave closure

These plots also use the absolute fitted normalization. There is **no**
`truth.sum()/fit.sum()` rescaling.


In [ ]:
def component_amplitude(model, name, data, values):
    comp = component_by_name(model, name)
    dynamics = jnp.asarray(comp.function(data, values))
    coefficient = jnp.asarray(comp.coefficient.value(values))
    scale = model._component_scale(comp, values)
    return coefficient * scale * dynamics


def coherent_group(model, names, data, values):
    total = 0.0j
    for name in names:
        total = total + component_amplitude(model, name, data, values)
    return jnp.asarray(total)


WAVES = {
    "P": ("rho770", "omega782", "rho1450"),
    "D": ("f2_1270",),
    "F": ("rho3_1690",),
}

validation = generator.generate_phase_space(700_000, seed=SEED + 501)

def min_pipi_mass(sample):
    return np.sqrt(
        np.minimum(np.asarray(sample.s13), np.asarray(sample.s23))
    )

def binned_wave_intensity(sample, generator_model, fitted_model, names, fit_values, bins):
    data = sample.as_dict()
    truth_amp = np.asarray(coherent_group(generator_model, names, data, {}))
    fit_amp = np.asarray(coherent_group(fitted_model, names, data, fit_values))
    mass = min_pipi_mass(sample)
    weights = np.asarray(sample.weights)
    truth_hist, edges = np.histogram(
        mass, bins=bins, weights=weights*np.abs(truth_amp)**2
    )
    fit_hist, _ = np.histogram(
        mass, bins=edges, weights=weights*np.abs(fit_amp)**2
    )
    centers = 0.5*(edges[:-1] + edges[1:])
    return centers, truth_hist, fit_hist

bins = np.linspace(2*M_PI, 3.0, 75)

for wave, names in WAVES.items():
    centers, truth_hist, fit_hist = binned_wave_intensity(
        validation, generator, fit_model, names, fit_values, bins
    )
    fig, ax = plt.subplots(figsize=(7.5, 4.8), constrained_layout=True)
    ax.step(centers, truth_hist, where="mid", label=f"{wave}-wave truth")
    ax.step(centers, fit_hist, where="mid", linestyle="--", label=f"{wave}-wave fit")
    ax.set_xlabel(r"$m_{\rm low}(\pi^+\pi^-)$ [GeV]")
    ax.set_ylabel("absolute arbitrary intensity")
    ax.set_title(f"{wave}-wave closure")
    ax.legend()
    plt.show()


## Higher-wave coefficient pulls


In [ ]:
rows = []
for name in ("rho770", "omega782", "rho1450", "f2_1270", "rho3_1690"):
    truth = TRUTH_COEFFICIENTS[name]
    for field in ("x", "y"):
        pname = f"{name}.{field}"
        parameter = next(
            (p for p in session.parameters if p.name == pname),
            None,
        )
        if parameter is None:
            continue
        fitted = fit_values[pname]
        error = 0.0 if parameter.fixed else float(result.errors[pname])
        pull = (
            np.nan
            if error == 0.0
            else (fitted - truth[field]) / error
        )
        rows.append({
            "component": name,
            "parameter": field,
            "truth": truth[field],
            "fit": fitted,
            "error": error,
            "pull": pull,
            "fixed": parameter.fixed,
        })

coefficient_table = pd.DataFrame(rows)
display(coefficient_table)


## Fit projections


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

session.plot_projection(
    result,
    "s13",
    bins=80,
    projection_size=300_000,
    projection_seed=SEED + 700,
    ax=axes[0],
)
session.plot_projection(
    result,
    "s23",
    bins=80,
    projection_size=300_000,
    projection_seed=SEED + 701,
    ax=axes[1],
)

plt.show()


## Optional diagnostics if the baseline closure fails

Run these only if the direct S-wave closure above is still poor.

The first check compares the JAX gradient used by Minuit with a finite-difference
gradient at the flat Cartesian QMI starting point, $1+0i$. The second evaluates
the NLL at the start and at the fitted point.


In [ ]:
# Uncomment if needed:
#
gradient_check = session.minimizer(
    tolerance=1e-4,
    verbose=1,
).check_gradient(
    start_values,
    step_scale=1e-5,
    print_table=True,
)

print("NLL(start) =", float(session.objective(start_values)))
print("NLL(fit)   =", float(session.objective(fit_values)))


## Interpretation

For this baseline test, a successful closure means:

1. the representation-only Cartesian QMI already follows the $\sigma$ truth closely;
2. the post-fit physical QMI magnitude and phase remain on the truth without
   any arbitrary rescaling or phase shift;
3. the Cartesian knot coordinates close within their uncertainties;
4. higher-wave coefficients stay statistically compatible with their generated values;
5. P/D/F-wave shapes and absolute intensities close;
6. the fitted event projections describe the generated toy.

This baseline uses a deliberately non-truth start: every QMI knot begins at
$1+0i$. Magnitude and phase are calculated only after the Cartesian fit.
